# Tutorial: Model Reduction and Control in STING

In this tutorial we show how STING can be used to construct a reduced-order model of the Western System Coordinating Council (WSCC) 9 bus test system. Using this reduced model we will then construct an output feedback controller to stabilize a problematic grid forming inverter. Finally, we validate that the controller is working properly via electromagnetic transient (EMT) simulation.


## Background
<img src="figures/wscc9.jpg" width="500">

**Fig 1**: Modified WSCC $230$-kV $9$-bus system.

**The problem:** The setting of this tutorial is a hypothetical interconnection study in the WSCC $230$-kV $9$-bus test system (Fig. 1). A utility is proposing to add a grid forming inverter (GFMI $_2$) and a transmission line to the grid. The utility provides the independent system operator (ISO) with a detailed model of the proposed project that can be used in EMT simulation. Using this model, the ISO determines that if interconnected the new inverter could cause grid instabilities. The ISO concludes that, as is, the proposed project should not be added to the grid. The utility is now tasked with redesigning the control mechanisms of their inverter.

This task however presents a challenge. The developer plans to construct a stabilizing output feedback controller for the inverter. To ensure that the inverter behaves properly when added to the grid, the developer would like to use a model of external grid's dynamics in their controller. While the ISO has a complete model of the system they cannot share it with the developer due to intellectual property concerns.

**Proposed solution:** The following pipeline is proposed to solve the described scenario:

1. To address the privacy concerns and reduce model complexity, the ISO constructs a reduced-order model of the system in Fig. 1, excluding proposed project. This reduced-order model is given to the interconnecting party.
2. The interconnecting party designs an output feedback controller for the proposed GFMI$_{2}$ using the reduced-order model. They study the stability of the closed-loop system, including synthesized controller, through eigenvalue analysis.
3. The interconnecting party then provides the ISO with an EMT model of the improved GFMI$_{2}$ with the output feedback controller. The ISO verifies the stability of the system with the proposed project through EMT simulation, using the full-order model of the grid.

Now we will implement this pipeline in STING!

<img src="figures/pipeline.jpg" width="700">

**Fig 2**: Solution workflow.



## Walk Through

Next we will walk through this pipeline in STING line-by-line. Feel free to run all cells in advance and explore the associated code or files in the `outputs` directory.

In [2]:
# Package imports
import os
import control as ct
import numpy as np
import polars as pl

# STING imports
from sting import main
from sting.modules.model_order_reduction.balanced_truncation import BalancedTruncation
from sting.utils.dynamical_systems import make_smooth_step
from sting.utils.transformations import abc2dq0


# Local .py scripts
from control_design import construct_controller
from wscc_9 import wscc_9
from plotting_tools import compare_timeseries, create_static_figures

# Current working directory
cwd = os.getcwd()
dir_outputs = os.path.join(cwd, "outputs")
# Local directory paths
dir_ssm = os.path.join(cwd, "outputs", "small_signal_model")
dir_rom = os.path.join(cwd, "outputs", "model_order_reduction")
dir_with_ctr = os.path.join(cwd, "outputs", "emt_with_control")
dir_without_ctr = os.path.join(cwd, "outputs", "emt_without_control")

### Constructing a System Model

To create our system model we will simply run
```python
system = wscc_9(case_directory=cwd)
```
This constructs the WSCC 9 bus system by calling a helper function in `wscc_9.py`. Inside this script we first create our buses, then lines, and finally generator components. For instance, the following code creates a grid following inverter (GFLI$_1$) at bus 5. 
```python
gfli_1 = GFLI16A(
        name="gfli_1", bus="bus_5", zone="external",
        # Power flow 
        minimum_active_power_MW=50, maximum_active_power_MW=100, 
        minimum_reactive_power_MVAR=-100, maximum_reactive_power_MVAR=100,
        cost_variable_USDperMWh=10, base_power_MVA=100, base_voltage_kV=0.48, 
        base_frequency_Hz=60,
        # LCL filter
        rf1_pu=0.002, xf1_pu=0.07, csh_pu=0.01, rsh_pu=100, 
        txr_power_MVA=100, txr_voltage1_kV=0.48, txr_voltage2_kV=230, 
        txr_r1_pu=0.003/2, txr_x1_pu=0.08/2, txr_r2_pu=0.003/2, 
        txr_x2_pu=0.08/2, 
        # Phase-locked loop (PLL)
        kp_pll_rad_s=100, ki_pll_rad2_s2=2500, tau_pll_s=1/100,
        # Inner current controller
        kp_cc_pu=0.05, ki_cc_puHz=0.6, kff_cc=0.75,
        # Power controllers
        kp_pc_pu=0.1, ki_pc_puHz=100
    )
```

After constructing all components we create a `System` object. This serves as a container for all components so that we can access them later.
```python
system = System(case_directory=case_directory)

for component in buses + timepoints + loads + lines + generators:
    system.add(component)

system.apply("post_system_init", system)
```

In [3]:
system = wscc_9(case_directory=cwd)

2026-09-02 11:32:42
   ______   __ 
  / __/ /__/ /_  _____ _
 _\ \/ __/ _// \/ / _ `/
/___/__///  /_/\_/\_, /
       /         /___/  

Version: 0.1.0 (2025-11-12)

> Initializing system ...
 System initialization completed.


### Building a Small-Signal Model

<img src="figures/ssm.jpg" width="700">

**Fig 3**: Solution workflow, constructing a small-signal model.

For control applications and analysis it is typically of interest to have a small-signal model. That is, the matrices $A$, $B$, $C$, $D$, such that

$$\tfrac{d}{dt}{\Delta x} = A \Delta x + B \Delta u$$
$$\Delta y = C \Delta x + D\Delta u$$


Building a small-signal model in STING is very easy! We simply execute the following line of code:
```python
system, ssm = main.run_ssm(system=system)
```

Internally STING will perform the following operations:
1. Solve AC power flow to find a steady-state equilibrium point, about which to linearize.
2. Each component internally computes its initial conditions and small-signal model using the power flow solution.
3. All of the of the component-level small-signal models are interconnected using the Component Connection Method [[DS81](#DS81)] to form a system-level small-signal model.

For more information on this process see [[SSH26](#SSH26)].

In [4]:
# Build a small signal model (SSM)
system, ssm = main.run_ssm(system=system)


>> Starting AC power flow...

Model settings: ModelSettings(generator_type_costs='linear', power_flow_formulation='polar', load_shedding=True, write_model_file=False)
Solver settings: SolverSettings(solver_name='ipopt', tee=True, solver_options=None)
> Initializing construction of the optimization model for capacity expansion ... 
 - Decision variables of active power and reactive power for generators
   Size: 8 variables
 - Constraints for generator active power dispatch limits
   Size: 4 constraints
 - Constraints for generator reactive power dispatch limits
   Size: 4 constraints
 - Expressions for dispatch of active power at any bus
   Size: 9 expressions
 - Expressions for dispatch of reactive power at any bus
   Size: 9 expressions
 - Expressions for generation cost per timepoint
   Size: 1 expressions
> Initializing construction of ac power flow variables, constraints, and costs ... 
 - Processing load data for power flow model
 - Decision variables of bus voltage magnitudes an

Now let's run a time domain simulation with the resulting small-signal model. First, we will need to construct an input vector $\Delta u(t)$. For the purposes of this tutorial we will step the voltage reference of the grid forming inverter by $0.1$ pu, relative to nomial.

In [5]:
# Create input signal to the proposed inverter project
step = make_smooth_step(step_time=0.1, initial_value=0.0, final_value=0.10, transient_width=5e-3)
inputs = {'gfmi_18a_0': {'v_ref': step}}
# Simulation length in seconds
t_max = 1.5

Having constructed our inputs we can send the differential equations to one of `scipy`'s solvers to obtain a time domain response.

In [6]:
# Run a time domain simulation of the SSM
ssm.simulate_ssm(inputs=inputs, t_max=t_max, output_directory=dir_ssm)

> Initializing construction and solution of differential equations associated to system-level small-signal model ... 
 - Writing SSM simulation results in /Users/adamsedlak/Documents/Python/PowerUp2026-Open-Source-Tools/STING/outputs/small_signal_model
 - Plotting SSM simulation results in /Users/adamsedlak/Documents/Python/PowerUp2026-Open-Source-Tools/STING/outputs/small_signal_model
> Completed in 0.93 seconds. 



### Model Reduction

<img src="figures/mor.jpg" width="700">

**Fig 4**: Solution workflow, constructing a reduced-order model.

Model reduction can be used to construct an approximate representation of a small-signal model with fewer states. For instance, we may hypothesize that the states $x \in \mathbb{R}^n$ can be reasonably approximated by some lower-order state vector $x_r \in \mathbb{R}^r$, where $r < n$. Stated equivalently, there exists a matrix $V \in \mathbb{R}^{n \times r}$ such that $x \approx V x_r$. If we can identify such a $V$ and its left inverse $W^\top$, such that $W^\top V = I_r$, we can *project* our state-space model into a lower dimension via

$$A_r = W^\top A V \quad \quad B_r = W^\top B$$
$$C_r =C V \quad \quad D_r = D$$


In this tutorial we will create a reduced order model of all components in the zone labeled  `"external"`. You can refer to `wscc_9.py` to see which components we labeled with `zone="external"` when they were instantiated. After constructing a reduced-order model we will interconnect it will the full-order model of all components in the proposed project. In this manner we are essentially creating a dynamic circuit equivalent model of the grid excluding the project. 

Here we will apply balanced truncation to removing the states that are both hard to control and observe. In code we assign a `BalancedTruncation` object to the `"external"` zone, specifying that the resulting model should have $33$ states.
```python
balanced_truncation = {
    "external": BalancedTruncation(r=33, method="truncate")
    }
rom = main.run_model_reduction(ssm=ssm, reductions=balanced_truncation)
```
Internally, STING will resolve which components should be reduced, based on their zone, and construct the projection matrices $V$ and $W$. For more information on this process see [[SSH27](#SSH27)].


In [7]:
balanced_truncation = {
    "external": BalancedTruncation(r=33, method="truncate")
    }
rom = main.run_model_reduction(ssm=ssm, reductions=balanced_truncation)

2026-09-02 11:32:43
   ______   __ 
  / __/ /__/ /_  _____ _
 _\ \/ __/ _// \/ / _ `/
/___/__///  /_/\_/\_, /
       /         /___/  

Version: 0.1.0 (2025-11-12)

> Initializing system ...
 System initialization completed.


Next we report some of the properties of the resulting reduced-order model using the python `control` library. In particular the $\mathcal{H}_2$ relative error, aims to capture the extent to which the dynamic response of the reduced-order model differs from the full-order model. 

Finally, we simulate the response of the reduced-order model when subjected to the same inputs.

In [8]:
# Compute statistics of the ROM and FOM
external_grid = rom.system.linear_subsystems[0]
ss_fom = ct.ss(*external_grid.full_order_model.data)
ss_rom = ct.ss(*external_grid.reduced_order_model.data)
print("Full-order model has", ss_fom.nstates, "states")
print("Reduced-order model (without proposed project) ", ss_rom.nstates, "states")
print("H_2 Error", round(100 * ct.norm(ss_fom - ss_rom,p=2) / ct.norm(ss_fom, p=2),3), "%")
print("Max eigenvalue of the ROM + study area: ", np.max(np.linalg.eigvals(rom.model.A).real))

Full-order model has 60 states
Reduced-order model (without proposed project)  33 states
H_2 Error 0.934 %
Max eigenvalue of the ROM + study area:  -0.40800653247170215


In [9]:
# Simulate the reduced-order model
rom.simulate_ssm(inputs=inputs, t_max=t_max, output_directory=dir_rom)

> Initializing construction and solution of differential equations associated to system-level small-signal model ... 
 - Writing SSM simulation results in /Users/adamsedlak/Documents/Python/PowerUp2026-Open-Source-Tools/STING/outputs/model_order_reduction
 - Plotting SSM simulation results in /Users/adamsedlak/Documents/Python/PowerUp2026-Open-Source-Tools/STING/outputs/model_order_reduction
> Completed in 0.51 seconds. 



### Control Design

<img src="figures/control.jpg" width="700">

**Fig 5**: Solution workflow, constructing a output feedback controller.

Using the reduced-order grid and proposed project models we will design a output feedback controller. STING does not have a built in control synthesis module so we will import one from our prior work [[SH26](#SH26)]. At a high-level we will use the state space matrices to construct some matrix $F$ that transforms outputs into inputs.

<img src="figures/control_F.jpg" width="250">

**Fig 6**: Output feedback controller placement.

If you want to learn more about exactly how $F$ is constructed feel free to look inside `control_design.py` where the function `construct_controller` is defined. 

In [10]:
F = construct_controller(rom)

+-------------------------------------------------------------------+
Synthesis of partial-state feedback control for a multi-agent system
+-------------------------------------------------------------------+ 

Number of states: 56
Number of inputs: 1
Number of agents: 1
SDP status: optimal
Obj. value: 12282600.979862725
Closed-loop system eigenvalues: 
shape: (56, 5)
┌───────────┬────────────┬──────────────────────┬──────────────────┬───────────────────────┐
│ real      ┆ imag       ┆ natural_frequency_hz ┆ damping_ratio_pu ┆ time_constant_seconds │
│ ---       ┆ ---        ┆ ---                  ┆ ---              ┆ ---                   │
│ f64       ┆ f64        ┆ f64                  ┆ f64              ┆ f64                   │
╞═══════════╪════════════╪══════════════════════╪══════════════════╪═══════════════════════╡
│ -3.108    ┆ 0.0        ┆ 0.495                ┆ 1.0              ┆ 0.3217                │
│ -6.346    ┆ 27.913     ┆ 4.556                ┆ 0.222            ┆ 0.

After obtaining a controller $F$ we will place it in closed-loop simulation by defining the function `output_feedback_control`. In simulation this function accesses the currents in the LCL filter ($i^\text{vsc} _{dq}$ and $i^\text{bus} _{dq}$) and the angular velocity ($\omega$) of GFMI$_2$ and compute the appropriate control response the in the inverters power setpoint ($p^\text{set}$). That is

$$ \Delta p^{\text{set}} = F [\Delta\omega \quad \Delta i^\text{vsc}_d \quad \Delta i^\text{vsc}_q \quad \Delta i^\text{bus}_d \quad \Delta i^\text{bus}_q]^\top$$

In [11]:
# Initial conditions in the LCL filter
w0 = 1
x0 = rom.system.gfmi_18a[0].lcl_filter.emt_init
y0 = np.array([w0, x0.i_vsc_d, x0.i_vsc_q, x0.i_bus_d, x0.i_bus_q])

def output_feedback_control(t: float, x: np.ndarray, id: dict):
    # Unpack the states of the GFM
    i_vsc_abc = (x[id['gfmi_18a_0']['i_vsc_'+p]] for p in ['a','b','c'])
    i_bus_abc = (x[id['gfmi_18a_0']['i_bus_'+p]] for p in ['a','b','c'])
    angle = x[id['gfmi_18a_0']['angle']]
    w = x[id['gfmi_18a_0']['w']]

    # Transform abc to dq0  
    i_vsc_d, i_vsc_q, _ = abc2dq0(*i_vsc_abc, angle)
    i_bus_d, i_bus_q, _ = abc2dq0(*i_bus_abc, angle)

    # Control action
    y = np.array([w, i_vsc_d, i_vsc_q, i_bus_d, i_bus_q])
    delta_u = F @ (y - y0)

    return delta_u[0]

Finally, to place this controller in closed loop simulation we need to create a new nested dictionary for simulation inputs. This is done as follows:

In [12]:
controller = {'gfmi_18a_0': {'v_ref': step, 'p_ref': output_feedback_control}}

### Results
Now we will validate that the controller is working as intended by using the full-order nonlinear EMT model. First we will run an EMT time domain simulation without a controller. 

These simulation can take a few minutes to run, depending on your computer.

In [13]:
# EMT simulation without control
main.run_emt(system=system, inputs=inputs, t_max=t_max, output_directory=dir_without_ctr)


>> Starting AC power flow...

Model settings: ModelSettings(generator_type_costs='linear', power_flow_formulation='polar', load_shedding=True, write_model_file=False)
Solver settings: SolverSettings(solver_name='ipopt', tee=True, solver_options=None)
> Initializing construction of the optimization model for capacity expansion ... 
 - Decision variables of active power and reactive power for generators
   Size: 8 variables
 - Constraints for generator active power dispatch limits
   Size: 4 constraints
 - Constraints for generator reactive power dispatch limits
   Size: 4 constraints
 - Expressions for dispatch of active power at any bus
   Size: 9 expressions
 - Expressions for dispatch of reactive power at any bus
   Size: 9 expressions
 - Expressions for generation cost per timepoint
   Size: 1 expressions
> Initializing construction of ac power flow variables, constraints, and costs ... 
 - Processing load data for power flow model
 - Decision variables of bus voltage magnitudes an

System: 
  - voltage_source_4a: 1
  - gfli_16a: 1
  - gfmi_18a: 2
  - buses: 9
  - loads: 3
  - constant_impedance_loads: 3
  - lines: 9
  - branch_series_rl: 9
  - shunt_parallel_rc: 9
  - timepoints: 1

Next we will run an EMT simulation with the controller. Note that now `inputs=controller` so the dynamic response to the same step input will be different.

In [14]:
# EMT simulation with control
main.run_emt(system=system, inputs=controller, t_max=t_max, output_directory=dir_with_ctr)


>> Starting AC power flow...

Model settings: ModelSettings(generator_type_costs='linear', power_flow_formulation='polar', load_shedding=True, write_model_file=False)
Solver settings: SolverSettings(solver_name='ipopt', tee=True, solver_options=None)
> Initializing construction of the optimization model for capacity expansion ... 
 - Decision variables of active power and reactive power for generators
   Size: 8 variables
 - Constraints for generator active power dispatch limits
   Size: 4 constraints
 - Constraints for generator reactive power dispatch limits
   Size: 4 constraints
 - Expressions for dispatch of active power at any bus
   Size: 9 expressions
 - Expressions for dispatch of reactive power at any bus
   Size: 9 expressions
 - Expressions for generation cost per timepoint
   Size: 1 expressions
> Initializing construction of ac power flow variables, constraints, and costs ... 
 - Processing load data for power flow model
 - Decision variables of bus voltage magnitudes an

System: 
  - voltage_source_4a: 1
  - gfli_16a: 1
  - gfmi_18a: 2
  - buses: 9
  - loads: 3
  - constant_impedance_loads: 3
  - lines: 9
  - branch_series_rl: 9
  - shunt_parallel_rc: 9
  - timepoints: 1

After running both scripts we can compare the outputs.

In [15]:
file = "gfmi_18a_0.csv"
cols_emt = ["w", "i_vsc_d", "i_vsc_q", "v_sh_d", "v_sh_q", "i_bus_d", "i_bus_q"]
cols_ssm = ["w", "i_vsc_d", "i_vsc_q", "v_lcl_sh_d", "v_lcl_sh_q", "i_bus_d", "i_bus_q"]

# EMT dynamics vs. small-signal model dynamics
compare_timeseries(
    df1=pl.read_csv(os.path.join(dir_without_ctr, file)),
    df2=pl.read_csv(os.path.join(dir_ssm, file)),
    left_to_right=dict(zip(cols_emt, cols_ssm)),
    df1_name="EMT",
    df2_name="SSM",
    figure_filepath=f"{cwd}/outputs/comparison_emt_vs_ssm.html",
    df1_color="#004488",
    df2_color="#6699CC"
)

# Small-signal model vs. reduced-order model
compare_timeseries(
    df1=pl.read_csv(os.path.join(dir_rom, file)),
    df2=pl.read_csv(os.path.join(dir_ssm, file)),
    left_to_right=dict(zip(cols_ssm, cols_ssm)),
    df2_name="Small-signal model",
    df1_name="Reduced-order model",
    figure_filepath=f"{cwd}/outputs/comparison_ssm_vs_rom.html",
    df2_color="#6699CC",
    df1_color="#EE99AA"
)

# EMT with control vs. EMT without control
compare_timeseries(
    df1=pl.read_csv(os.path.join(dir_without_ctr, file)),
    df2=pl.read_csv(os.path.join(dir_with_ctr, file)),
    left_to_right=dict(zip(cols_emt, cols_emt)),
    df1_name="Without control",
    df2_name="With control",
    figure_filepath=f"{cwd}/outputs/comparison_woc_vs_wc.html",
    df1_color="#004488",
    df2_color="#DDAA33"
)


create_static_figures(dir_outputs)

## References

<a id="DS81"></a>[DS81] R. DeCarlo and R. Saeks, *Interconnected Dynamical Systems*. New York
Dekker, 1981.



<a id="SH26"></a> [SH26] P. Serna-Torre and P. Hidalgo-Gonzalez, “Static output feedback control for multi-agent systems using nash equilibrium,” Under review, 2026.

<a id="SSH26"></a>[SSH26] P. Serna-Torre, A. Sedlak, and P. Hidalgo-Gonzalez. "A generalized and open-source state-space framework to derive small-signal models for EMT dynamics of inverter-dominated grids", *TechRxiv*, 2026.

<a id="SSH27"></a>[SSH27] A. Sedlak, P. Serna-Torre, and P. Hidalgo-Gonzalez, “Model reduction of electromagnetic transient dynamics for inverter-based grids: an interconnected systems framework,” *Electric Power Systems Research*, vol. 262, p. 113671, 2027. [Online]. Available: https://www.sciencedirect.com/science/article/pii/S0378779626009648.